In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/cleaned_telco_data.csv")
df.head(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  Paymen

In [4]:
df.shape

(7032, 21)

In [9]:
# Create target variable
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

- Yes → 1 → Customer Churned
- No → 0 → Customer Retained

In [10]:
# Seperate categorical columns
categorical_cols = df.select_dtypes(
    include="object"
).columns

categorical_cols

C:\Users\Asus\AppData\Local\Temp\ipykernel_19116\2830024089.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [11]:
# Removing customeID column
df.drop(
    columns=["customerID"],
    inplace=True
)

**7590-VHVEG** - This value gives no business meaning. Models should never memorize IDs.

In [12]:
# Binary Columns
binary_cols = [
    col for col in df.columns
    if df[col].nunique() == 2
]

binary_cols

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'PhoneService',
 'PaperlessBilling',
 'Churn']

In [13]:
# Encoding binary columns
binary_mapping = {
    "Yes": 1,
    "No": 0,
    "Female": 1,
    "Male": 0
}

binary_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_cols:
    df[col] = df[col].map(binary_mapping)

**Binary categorical columns** were encoded into `0` and `1` values to make them suitable for machine learning model training.

In [14]:
# Handle Multi-category Columns (One-Hot Encoding)
df = pd.get_dummies(
    df,
    columns=[
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaymentMethod"
    ],
    drop_first=True
)

### One-Hot Encoded Features Interpretation

| Original Feature | Generated Columns | Interpretation |
|------------------|-------------------|----------------|
| MultipleLines | MultipleLines_No phone service, MultipleLines_Yes | `(0,0)` → No, `(0,1)` → No phone service, `(1,0)` → Yes |
| InternetService | InternetService_Fiber optic, InternetService_No | `(0,0)` → DSL, `(1,0)` → Fiber optic, `(0,1)` → No internet |
| OnlineSecurity | OnlineSecurity_No internet service, OnlineSecurity_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| OnlineBackup | OnlineBackup_No internet service, OnlineBackup_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| DeviceProtection | DeviceProtection_No internet service, DeviceProtection_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| TechSupport | TechSupport_No internet service, TechSupport_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| StreamingTV | StreamingTV_No internet service, StreamingTV_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| StreamingMovies | StreamingMovies_No internet service, StreamingMovies_Yes | `(0,0)` → No, `(0,1)` → No internet service, `(1,0)` → Yes |
| Contract | Contract_One year, Contract_Two year | `(0,0)` → Month-to-month, `(1,0)` → One year, `(0,1)` → Two year |
| PaymentMethod | PaymentMethod_Credit card (automatic), PaymentMethod_Electronic check, PaymentMethod_Mailed check | `(0,0,0)` → Bank transfer (automatic), remaining combinations represent their respective payment methods |

***Explanation:*** *Multi-category features were transformed into numerical dummy variables using One-Hot Encoding. One category from each feature was treated as the baseline (`drop_first=True`) and is automatically represented when all generated dummy columns have a value of `0`. This avoids redundant information and prevents multicollinearity while preserving all category information for machine learning models.*

In [15]:
df.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,0,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,False,False,True,False,False,False,True
2,0,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True


In [17]:
# Convert bool columns to 0 and 1 value
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

In [18]:
df.head(1)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,0,0,1,0


Boolean features generated during One-Hot Encoding were converted from `True` and `False` to `1` and `0`, respectively, to maintain a fully numerical dataset suitable for machine learning algorithms.

- `0` → False (Category not present)
- `1` → True (Category present)

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   PaperlessBilling                       7032 non-null   int64  
 7   MonthlyCharges                         7032 non-null   float64
 8   TotalCharges                           7032 non-null   float64
 9   Churn                                  7032 non-null   int64  
 10  MultipleLines_No phone service         7032 non-null   int64  
 11  MultipleLines_Y

In [20]:
df.shape

(7032, 31)

In [21]:
df.isnull().sum().sum()

np.int64(0)

In [23]:
# Saving Final Feature Engineered Dataset
df.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!


*The feature-engineered dataset was saved as `feature_engineered_data.csv` in the `data/processed` directory. This dataset will be used in the next phase for train-test splitting, feature scaling, and machine learning model development.*

# Conclusion

The dataset was successfully transformed into a fully numerical, machine-learning-ready format by encoding binary and multi-category features, removing identifier columns, and validating data quality. The final dataset contains **7,032 customer records and 31 features** and is now ready for model preparation and predictive modelling.